# 📖 Story Generation Pipeline — Google Colab

**Four-stage pipeline:** User Prompt → MoPS Premise → DOME Memory → T5 Outline → BART Story

| Stage | Module | Model |
|-------|--------|-------|
| 1 | Premise expansion (MoPS-inspired) | Template-based |
| 2 | Memory extraction (DOME-inspired) | spaCy NER |
| 3 | Outline generation (EtriCA-inspired) | T5-small |
| 4 | Story generation (Hierarchical) | BART-base |

### Notebook sections
1. ⚙️ Setup — clone repo & install dependencies
2. 📊 Data Preparation (ROCStories + WritingPrompts)
3. 🏋️ Training (ROCStories + WritingPrompts)
4. 📏 Comprehensive Evaluation (ROUGE-L / BLEU / METEOR / BERTScore)
5. 🔬 Ablation Study (6 conditions)
6. 🎨 Interactive Story Generation Demo
7. 💾 Download Results

---
> **Before you start:** `Runtime → Change runtime type → T4 GPU`  
> **Note:** Colab sessions are ephemeral. Trained models and processed data are stored in `/content/` and will be lost when the session ends. Download anything you want to keep (Section 7).

---
## ⚙️ 1. Setup

Run these three cells at the start of every session.

In [ ]:
# ── 1a. Check GPU ─────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU : {gpu}")
    print(f"VRAM: {vram:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > GPU (T4)")

In [ ]:
# ── 1b. Clone repository from GitHub ─────────────────────────────────────
GITHUB_REPO = "https://github.com/Ardameliksah/StoryGeneration.git"
BRANCH      = "mert_test"   # <- branch where all code lives
REPO_DIR    = "/content/StoryGeneration"

import os, pathlib

if pathlib.Path(REPO_DIR).exists():
    # Already cloned — pull latest on the correct branch
    !git -C {REPO_DIR} fetch --quiet
    !git -C {REPO_DIR} checkout --quiet {BRANCH}
    !git -C {REPO_DIR} pull --quiet
    print(f"Repo updated ({BRANCH}) at {REPO_DIR}")
else:
    !git clone --quiet --branch {BRANCH} {GITHUB_REPO} {REPO_DIR}
    print(f"Repo cloned ({BRANCH}) to {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Verify required source files
REQUIRED = [
    "inference.py", "metrics.py", "evaluate.py", "ablation.py",
    "train_outline.py", "train_story.py",
    "prepare_data.py", "prepare_writingprompts.py",
]
missing = [f for f in REQUIRED if not pathlib.Path(f).exists()]
if missing:
    print("MISSING files:", missing)
else:
    print("All source files present")

In [ ]:
# ── 1c. Install dependencies (~2–3 min, run once per session) ─────────────
!pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.30.0 \
    sentencepiece>=0.2.0 \
    rouge-score>=0.1.2 \
    sacrebleu>=2.4.0 \
    nltk>=3.8.0 \
    bert-score>=0.3.13 \
    spacy>=3.7.0 \
    datasets>=2.19.0

!python -m spacy download en_core_web_sm -q

import nltk
for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

print("All dependencies installed")

---
## 📊 2. Data Preparation

### 2a. ROCStories

ROCStories requires a manual upload because the dataset is not publicly downloadable via an API.  
Upload `rocstories_train.txt` and `rocstories_test.txt` using the cell below.

In [ ]:
# ── 2a-i. Upload ROCStories raw files ────────────────────────────────────
import os, pathlib
from google.colab import files

os.makedirs("data/raw", exist_ok=True)

train_ok = pathlib.Path("data/raw/rocstories_train.txt").exists()
test_ok  = pathlib.Path("data/raw/rocstories_test.txt").exists()

if train_ok and test_ok:
    print("ROCStories files already present — skipping upload")
else:
    print("Select both rocstories_train.txt and rocstories_test.txt ...")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = pathlib.Path("data/raw") / fname
        dest.write_bytes(data)
        print(f"  Saved -> {dest}")

In [ ]:
# ── 2a-ii. Build ROCStories JSONL files ───────────────────────────────────
# Output: data/processed/outline_{train,val,test}.jsonl
#         data/processed/story_{train,val,test}.jsonl
import pathlib

if pathlib.Path("data/processed/story_train.jsonl").exists():
    print("Processed ROCStories data already exists — skipping")
else:
    !python prepare_data.py

### 2b. WritingPrompts *(optional — only needed for WP fine-tuning)*

Downloads `euclaise/writingprompts` from HuggingFace automatically.  
Full dataset ~272K stories (~20 min). Use `MAX_WP_STORIES` to limit for quick experiments.

In [ ]:
# ── 2b. Build WritingPrompts JSONL files ──────────────────────────────────
import pathlib

MAX_WP_STORIES = 50000   # None = full ~272K dataset

if pathlib.Path("data/processed/wp_story_train.jsonl").exists():
    print("Processed WritingPrompts data already exists — skipping")
else:
    flag = f"--max-stories {MAX_WP_STORIES}" if MAX_WP_STORIES else ""
    !python prepare_writingprompts.py {flag}

---
## 🏋️ 3. Training

### 3a. T5-small — Outline Generator (ROCStories)

Learns: `story title → event1 | event2 | event3`  
~15 min / epoch on T4.

In [ ]:
# ── Train from scratch (3 epochs recommended) ─────────────────────────────
!python train_outline.py --data roc --epochs 3

### 3b. BART-base — Story Generator (ROCStories)

Learns: `title outline: [events] [MEM] ... → full story`  
~25 min / epoch on T4.

In [ ]:
# ── Train from scratch (3 epochs recommended) ─────────────────────────────
!python train_story.py --data roc --epochs 3

### 3c. WritingPrompts Fine-tuning *(optional)*

`--grad-accum 4` simulates batch size 32 on T4 (WP stories are longer, so the real batch must be smaller).

In [ ]:
# ── Train T5 outline on WritingPrompts ────────────────────────────────────
!python train_outline.py --data wp --epochs 3 --grad-accum 4

# ── Train BART story on WritingPrompts ────────────────────────────────────
!python train_story.py --data wp --epochs 3 --grad-accum 4

---
## 📏 4. Comprehensive Evaluation

| Metric | What it captures |
|--------|------------------|
| **ROUGE-L** | Longest common subsequence overlap |
| **BLEU** | n-gram precision (sacrebleu) |
| **METEOR** | Precision + recall with stemming & synonym matching |
| **BERTScore** | Semantic similarity via contextual embeddings |

> Low BLEU/ROUGE is expected for open-ended story generation.

In [ ]:
# ── ROCStories — validation split ────────────────────────────────────────
!python evaluate.py --data roc --n 500 --split val

In [ ]:
# ── ROCStories — test split (final / paper numbers) ───────────────────────
!python evaluate.py --data roc --n 500 --split test

In [ ]:
# ── WritingPrompts — validation split (requires Section 3c) ──────────────
!python evaluate.py --data wp --n 200 --split val

In [ ]:
# ── Python API — returns a dict, useful for building tables ───────────────
import importlib, sys
sys.path.insert(0, "/content/StoryGeneration")

import evaluate as eval_mod
importlib.reload(eval_mod)

roc_val = eval_mod.evaluate(
    data="roc", split="val", n=100,
    use_memory=True,
    bert_model="distilbert-base-uncased",
)
print(roc_val)

---
## 🔬 5. Ablation Study

Six conditions isolate the contribution of each pipeline component:

| Condition | What changes | Tests |
|-----------|-------------|-------|
| `full` | — baseline — | — |
| `no_memory` | DOME `[MEM]` block removed | memory module |
| `no_outline` | no outline fed to BART | outline module |
| `with_premise` | premise text added to BART input | premise contribution |
| `sent_outline` | oracle sentences from reference as outline | event vs. sentence format + ceiling |
| `wp_models` | WP-trained models evaluated on WP val | dataset effect |

In [ ]:
# ── Quick smoke test (50 examples, ~10 min on T4) ─────────────────────────
!python ablation.py --n 50

In [ ]:
# ── Full ablation (200 examples per condition, ~40 min on T4) ─────────────
!python ablation.py --n 200 --split val

In [ ]:
# ── Python API — run ablation and render a highlighted DataFrame ───────────
import importlib, sys
sys.path.insert(0, "/content/StoryGeneration")

import ablation as ab
importlib.reload(ab)

import pandas as pd

abl_results = ab.ablation(
    n=100,
    split="val",
    bert_model="distilbert-base-uncased",
)

df = pd.DataFrame(abl_results).T
df.index.name = "condition"
df.columns    = ["ROUGE-L", "BLEU", "METEOR", "BERTScore"]
df = df.round(4)
display(df.style.highlight_max(axis=0, color="#c6efce"))

---
## 🎨 6. Interactive Story Generation Demo

In [ ]:
# ── Generate a story from a custom prompt ────────────────────────────────
import sys
sys.path.insert(0, "/content/StoryGeneration")

import inference

# 'roc' = ROCStories models  |  'wp' = WritingPrompts models
MODEL = "roc"

inference.T5_CHECKPOINT   = f"models/t5_outline{'_wp' if MODEL == 'wp' else ''}"
inference.BART_CHECKPOINT = f"models/bart_story{'_wp' if MODEL == 'wp' else ''}"
inference._outline_cache  = None
inference._story_cache    = None

PROMPT = "She finally found what she had been looking for"   # <- change me

result = inference.run_pipeline(PROMPT)

In [ ]:
# ── Side-by-side comparison of pipeline variants ─────────────────────────
import inference
from inference import generate_outline, generate_story, extract_memory, expand_premise

PROMPT = "The last train left without him"   # <- change me

outline = generate_outline(PROMPT)
memory  = extract_memory(PROMPT)
premise = expand_premise(PROMPT)

print(f"Prompt  : {PROMPT}")
print(f"Outline : {outline}")
print(f"Memory  : {memory or '(none extracted)'}")
print()

variants = {
    "Full pipeline" : dict(use_memory=True,  use_outline=True,  use_premise=False),
    "No memory"     : dict(use_memory=False, use_outline=True,  use_premise=False),
    "No outline"    : dict(use_memory=False, use_outline=False, use_premise=False),
    "With premise"  : dict(use_memory=True,  use_outline=True,  use_premise=True),
}

for label, flags in variants.items():
    story = generate_story(PROMPT, outline, memory, premise=premise, **flags)
    print(f"── {label} ──")
    print(story)
    print()

In [ ]:
# ── Batch generation ──────────────────────────────────────────────────────
import inference
from inference import generate_outline, generate_story, extract_memory

inference._outline_cache = None
inference._story_cache   = None

prompts = [
    "She finally found what she had been looking for",
    "The dog had been waiting at the door for three days",
    "He opened the letter and his hands began to shake",
    "The old library held a secret no one had discovered",
    "They said it was impossible, but she proved them wrong",
]

stories = []
for p in prompts:
    outline = generate_outline(p)
    memory  = extract_memory(p)
    story   = generate_story(p, outline, memory)
    stories.append({"prompt": p, "outline": outline, "story": story})
    print(f"done: {p[:55]}")

# Show sample
s = stories[0]
print(f"\nPrompt  : {s['prompt']}")
print(f"Outline : {s['outline']}")
print(f"Story   : {s['story']}")

In [ ]:
# ── Score a hypothesis against a reference ────────────────────────────────
import metrics

hypothesis = "She searched for years until she stumbled upon a small shop. Inside was the necklace her grandmother had lost. The shopkeeper smiled as if he had been expecting her. She paid with the last of her savings and walked home in tears."
reference  = "She had been looking for the missing heirloom her whole life. One afternoon she found it in an antique market downtown. The seller told her it had been there for twenty years."

scores = metrics.compute_all([hypothesis], [reference], verbose=True)
print(scores)

---
## 💾 7. Download Results

Runs a full evaluation + ablation and downloads JSON/CSV files to your computer.

In [ ]:
import importlib, json, pathlib, sys
import pandas as pd
from google.colab import files

sys.path.insert(0, "/content/StoryGeneration")

# ── Evaluation ────────────────────────────────────────────────────────────
import evaluate as eval_mod
importlib.reload(eval_mod)
roc_scores = eval_mod.evaluate(data="roc", split="val", n=500)

# ── Ablation ──────────────────────────────────────────────────────────────
import ablation as ab
importlib.reload(ab)
abl_scores = ab.ablation(n=200, split="val")

# ── Save files ────────────────────────────────────────────────────────────
out = pathlib.Path("results")
out.mkdir(exist_ok=True)

(out / "evaluation_roc.json").write_text(json.dumps(roc_scores, indent=2))
(out / "ablation.json").write_text(json.dumps(abl_scores, indent=2))

df_eval = pd.DataFrame([roc_scores], index=["ROCStories"])
df_abl  = pd.DataFrame(abl_scores).T
df_eval.to_csv(out / "evaluation_roc.csv")
df_abl.to_csv(out / "ablation.csv")

# ── Display ───────────────────────────────────────────────────────────────
print("=== Evaluation ===")
display(df_eval.round(4))
print("\n=== Ablation ===")
display(df_abl.round(4).style.highlight_max(axis=0, color="#c6efce"))

# ── Download to local machine ─────────────────────────────────────────────
for f in out.glob("*"):
    files.download(str(f))
    print(f"Downloading {f.name}")

In [ ]:
# ── Download trained model checkpoints (optional) ─────────────────────────
# Zips models/t5_outline and models/bart_story, then triggers browser download
import pathlib
from google.colab import files

MODELS_TO_DOWNLOAD = [
    ("models/t5_outline",   "t5_outline_roc.zip"),
    ("models/bart_story",   "bart_story_roc.zip"),
    # ("models/t5_outline_wp",  "t5_outline_wp.zip"),   # uncomment for WP
    # ("models/bart_story_wp",  "bart_story_wp.zip"),
]

for src, zip_name in MODELS_TO_DOWNLOAD:
    if pathlib.Path(src).exists():
        !zip -r {zip_name} {src} -q
        files.download(zip_name)
        print(f"Downloading {zip_name}")
    else:
        print(f"Skipping {src} — not found (not trained yet?)")